# Multi-Layer Perceptron

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt
import time


In [4]:
df = pd.read_csv("datasets/airlines_delay.csv")
df_sample = df.sample(n=15000, random_state=1)
cat_cols = ['Airline','AirportFrom','AirportTo']
cont_cols = ['Time','Length']
encoder = OneHotEncoder(sparse_output=False)
cat_data = encoder.fit_transform(df_sample[cat_cols])
scaler = MinMaxScaler()
cont_data = scaler.fit_transform(df_sample[cont_cols])
X = np.hstack([cat_data, cont_data, df_sample['DayOfWeek'].values.reshape(-1,1)]).T
Y = df_sample['Class'].values.reshape(1,-1)
X_train, X_test, Y_train, Y_test = train_test_split(X.T, Y.T, test_size=0.2, random_state=1)
X_train, X_test, Y_train, Y_test = X_train.T, X_test.T, Y_train.T, Y_test.T


In [5]:
def initialize_parameters(input_size, hidden_size1, hidden_size2, output_size):
    W1 = np.random.randn(hidden_size1, input_size) * 0.01
    b1 = np.zeros((hidden_size1,1))
    W2 = np.random.randn(hidden_size2, hidden_size1) * 0.01
    b2 = np.zeros((hidden_size2,1))
    W3 = np.random.randn(output_size, hidden_size2) * 0.01
    b3 = np.zeros((output_size,1))
    return {"W1":W1,"b1":b1,"W2":W2,"b2":b2,"W3":W3,"b3":b3}

def sigmoid(x):
    return 1/(1+np.exp(-x))

def sigmoid_derivative(x):
    return x*(1-x)

def forward_propagation(X, params):
    W1,b1,W2,b2,W3,b3 = params['W1'],params['b1'],params['W2'],params['b2'],params['W3'],params['b3']
    Z1 = W1.dot(X) + b1
    A1 = sigmoid(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = sigmoid(Z2)
    Z3 = W3.dot(A2) + b3
    A3 = sigmoid(Z3)
    cache = {"A1":A1,"A2":A2,"A3":A3}
    return A3, cache

def backward_propagation(X, Y, cache, params):
    m = X.shape[1]
    A1,A2,A3 = cache['A1'], cache['A2'], cache['A3']
    W2,W3 = params['W2'], params['W3']
    dZ3 = A3 - Y
    dW3 = 1/m * dZ3.dot(A2.T)
    db3 = 1/m * np.sum(dZ3, axis=1, keepdims=True)
    dZ2 = W3.T.dot(dZ3) * sigmoid_derivative(A2)
    dW2 = 1/m * dZ2.dot(A1.T)
    db2 = 1/m * np.sum(dZ2, axis=1, keepdims=True)
    dZ1 = W2.T.dot(dZ2) * sigmoid_derivative(A1)
    dW1 = 1/m * dZ1.dot(X.T)
    db1 = 1/m * np.sum(dZ1, axis=1, keepdims=True)
    return {"dW1":dW1,"db1":db1,"dW2":dW2,"db2":db2,"dW3":dW3,"db3":db3}

def update_parameters(params, grads, lr):
    for k in ['W1','b1','W2','b2','W3','b3']:
        params[k] -= lr * grads['d'+k]
    return params

def train_mlp(X, Y, input_size, hidden_size1, hidden_size2, output_size, num_iterations, lr):
    params = initialize_parameters(input_size, hidden_size1, hidden_size2, output_size)
    for _ in range(num_iterations):
        A3, cache = forward_propagation(X, params)
        grads = backward_propagation(X, Y, cache, params)
        params = update_parameters(params, grads, lr)
    return params


In [6]:
input_size = X.shape[0]
hidden_size1 = 32
hidden_size2 = 32
output_size = 1
num_iterations = 500
learning_rate = 0.01
start = time.time()
params = train_mlp(X_train, Y_train, input_size, hidden_size1, hidden_size2, output_size, num_iterations, learning_rate)
pred_train,_ = forward_propagation(X_train, params)
pred_test,_ = forward_propagation(X_test, params)
end = time.time()
acc_train = np.mean(np.round(pred_train)==Y_train)
acc_test = np.mean(np.round(pred_test)==Y_test)
print('Training accuracy:', acc_train)
print('Test accuracy:', acc_test)
print('Execution time:', end-start, 'seconds')


Training accuracy: 0.5530833333333334
Test accuracy: 0.5613333333333334
Execution time: 8.816011905670166 seconds
